# 08.03 Datasets & Evaluation — 트레이스에서 자동 평가 루프로

트레이스는 "지금 어떻게 굴러가는지" 를 보여주고, 평가는 "프롬프트 · 모델 · 코드를 바꿨을 때 더 좋아졌는지" 를 답한다. LangSmith 에서는 **트레이스를 그대로 데이터셋으로 끌어올리고**, 그 위에 code evaluator · LLM-as-judge · pairwise · summary 평가를 돌린다.

## 학습 목표

- `client.create_dataset` + `client.create_examples` 로 데이터셋을 만들고 예시를 추가한다
- 프로덕션 trace 를 `client.add_runs_to_dataset` 로 데이터셋에 이관한다
- Code evaluator 를 `def my_eval(inputs, outputs, reference_outputs) -> dict` 형식으로 작성한다
- LLM-as-judge evaluator 를 구조적 score 로 돌린다
- Pairwise / summary evaluator 로 두 실험 비교와 데이터셋 수준 지표를 낸다
- `from langsmith.evaluation import evaluate` 러너로 experiment 를 실행하고 이름을 붙인다
- 프로덕션 trace 에 **online evaluator** 를 자동 적용하는 흐름을 이해한다

## 사전 준비

`.env` 에 LangSmith 키 3종과 OpenAI 키가 있어야 한다.

```dotenv
LANGSMITH_API_KEY=lsv2_pt_...
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=langsmith-eval-demo
OPENAI_API_KEY=sk-...
```

평가를 돌리는 동안 experiment 프로젝트가 자동으로 분리돼 기록되므로, 프로덕션 프로젝트와 섞이지 않는다.

In [ ]:
# !pip install -U langsmith langchain langchain-openai

from dotenv import load_dotenv
import os
load_dotenv(override=True)

assert os.environ.get("LANGSMITH_API_KEY"), "LANGSMITH_API_KEY 미설정"
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 미설정"
os.environ.setdefault("LANGSMITH_PROJECT", "langsmith-eval-demo")
print("프로젝트:", os.environ["LANGSMITH_PROJECT"])

## 8.03.1 Dataset 생성 — 수동 예시 추가

도메인 전문가가 골든 Q&A 를 직접 적어서 `create_examples` 로 넣는다. `inputs` 와 `outputs` 는 모두 dict. `outputs` 는 reference 정답으로 code evaluator / LLM-as-judge 의 비교 대상이 된다.

<!-- phase-c:embed -->
![Datasets 목록](../assets/images/langsmith/03_datasets_and_evaluation/01_datasets_list.png)

*Datasets & Experiments 리스트 — `weather-bot-qa`(3 examples · 2 experiments)와 `agent-golden-traces`(영구 보존용 우수 트레이스) 두 개. Type 컬럼의 `kv` 는 key-value 구조 dataset 표시.*

In [ ]:
from langsmith import Client

client = Client()
DATASET_NAME = "weather-bot-qa"

try:
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description="날씨 봇 골든 Q&A — 도시명 정규화·응답 톤 검증용",
    )
except Exception:
    dataset = client.read_dataset(dataset_name=DATASET_NAME)

client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {"inputs": {"question": "서울 날씨 알려줘"},        "outputs": {"city": "서울"}},
        {"inputs": {"question": "부산광역시 날씨는?"},     "outputs": {"city": "부산"}},
        {"inputs": {"question": "오늘 제주도 날씨 어때요?"}, "outputs": {"city": "제주"}},
    ],
)
print(f"dataset 준비 완료: {dataset.name} ({dataset.id})")

## 8.03.2 프로덕션 trace 를 데이터셋으로 이관

수동 작성은 초기 시드에만 쓰고, 실제 규모는 **프로덕션 trace** 에서 온다. `client.add_runs_to_dataset` 로 run 의 `inputs`/`outputs` 가 그대로 example 로 복사된다. 실제 운영에서는 Annotation Queue 로 사람이 리뷰한 run 만 올리는 게 일반적이다.

<!-- phase-c:embed -->
![Dataset Examples 탭](../assets/images/langsmith/03_datasets_and_evaluation/03_dataset_examples_tab.png)

*`weather-bot-qa` 의 Examples 탭 — `client.create_examples(...)` 로 업로드한 3개 예제. Inputs 컬럼에 한국어 질문, Reference Outputs 컬럼에 기대 도시명. JSON / YAML 포맷 전환 토글, Splits 컬럼(학습/검증/테스트 분리) 제공.*

In [ ]:
# 현재 프로젝트에서 루트 run 몇 개를 골라 데이터셋에 이관
# langsmith 0.7.x 부터 add_runs_to_dataset 가 제거됨. create_examples 로 변환해서 추가.
root_runs = list(client.list_runs(
    project_name=os.environ["LANGSMITH_PROJECT"],
    is_root=True,
    limit=5,
))

examples = [
    {"inputs": r.inputs, "outputs": r.outputs, "metadata": {"source_run_id": str(r.id)}}
    for r in root_runs if r.outputs
]
if examples:
    client.create_examples(dataset_id=dataset.id, examples=examples)
print(f"{len(examples)} 개 run 을 {DATASET_NAME} 에 이관")

## 8.03.3 평가 대상 + Code evaluator

예제용으로 "질문에서 도시명을 뽑는" LLM 함수를 대상(target) 으로 삼는다. target 은 `inputs: dict -> outputs: dict` 형태면 된다.

Evaluator 는 **`(inputs, outputs, reference_outputs)`** 를 받아 `{"key": ..., "score": ...}` dict 를 반환한다. 결정적 휴리스틱은 비용 0, 지연 ~0 ms — 가능한 한 많이 넣는 게 이득.

<!-- phase-c:embed -->
![Evaluator 템플릿 갤러리](../assets/images/langsmith/03_datasets_and_evaluation/04_dataset_evaluators_tab.png)

*Evaluators 페이지 — 템플릿 8종(PII Leakage · Prompt Injection · Toxicity · Bias & Fairness · Hallucination · Correctness · Perceived Error · User Satisfaction) + 직접 작성(LLM-as-a-Judge · Code Evaluator).*

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

def extract_city(inputs: dict) -> dict:
    """질문에서 도시명만 뽑아낸다 (광역시/특별시 접미사 제거)."""
    q = inputs["question"]
    msg = llm.invoke(f"다음 질문에서 도시명만 한 단어로 답하세요 (예: 서울, 부산, 제주). 질문: {q}")
    return {"city": msg.content.strip()}

def city_exact_match(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """예측한 도시가 reference 와 정확히 일치하는가."""
    return {
        "key": "city_exact_match",
        "score": int(outputs.get("city") == reference_outputs.get("city")),
    }

def city_non_empty(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """빈 응답 방어 — 실패 모드 모니터링용."""
    return {"key": "city_non_empty", "score": int(bool(outputs.get("city", "").strip()))}

print(extract_city({"question": "부산광역시 날씨는?"}))

## 8.03.4 LLM-as-judge evaluator

정답 문자열이 없거나 자연어 품질(톤·정확성·도움됨)을 재야 할 때 쓴다. 같은 LLM 을 호출하지만 **출력을 구조화된 score 로 강제**하는 것이 핵심.

In [ ]:
from pydantic import BaseModel, Field

class Judgement(BaseModel):
    score: int = Field(..., ge=0, le=1, description="1이면 응답이 정답과 의미적으로 같음")
    reason: str

judge_llm = llm.with_structured_output(Judgement)

def semantic_city_match(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    verdict: Judgement = judge_llm.invoke(
        "질문과 기대되는 정답 도시, 모델이 뽑은 도시를 비교하여 의미적으로 같은지 판단하세요.\n"
        f"질문: {inputs['question']}\n기대: {reference_outputs.get('city')}\n모델: {outputs.get('city')}"
    )
    return {"key": "semantic_city_match", "score": verdict.score, "comment": verdict.reason}

## 8.03.5 `evaluate` 러너 + experiment 이름

`from langsmith.evaluation import evaluate` 가 표준 러너다. `experiment_prefix` 에 의미 있는 이름을 주면 UI 의 Experiments 뷰에서 바로 비교 가능.

<!-- phase-c:embed -->
![Pairwise Experiments 탭](../assets/images/langsmith/03_datasets_and_evaluation/05_pairwise_experiments_tab.png)

*Pairwise Experiments 탭 — 두 experiment 의 output 을 나란히 비교. `evaluate_comparative` API 로 실행 (2026-04 기준 session name 전체가 필요하므로 본 노트북의 `evaluate_comparative` 호출은 drift 로 실패, 빈 상태로 표시됨).*

In [ ]:
from langsmith.evaluation import evaluate

results = evaluate(
    extract_city,
    data=DATASET_NAME,
    evaluators=[city_exact_match, city_non_empty, semantic_city_match],
    experiment_prefix="city-extractor:gpt-4.1-mini",
    description="베이스라인 — 온도 0, 단일 프롬프트",
    max_concurrency=4,
    metadata={"model": "gpt-4.1-mini", "temperature": 0},
)
print("Experiment URL ->", results)

## 8.03.6 Pairwise + Summary evaluator

- **Pairwise**: 두 experiment 의 같은 example 출력을 놓고 "어느 쪽이 더 나은가" 를 뽑는다. A/B 프롬프트 실험에 쓴다. `evaluate_comparative` 러너.
- **Summary**: 데이터셋 전체 수준의 지표(예: 정확도 매크로 평균). 시그니처가 run/example 의 **리스트**를 받도록 다르다.

In [ ]:
from langsmith.evaluation import evaluate_comparative

# 1) 비교용 두번째 experiment (다른 프롬프트)
def extract_city_v2(inputs: dict) -> dict:
    q = inputs["question"]
    msg = llm.invoke(f"질문에서 오직 도시명 하나만, 접미사 없이 반환: {q}")
    return {"city": msg.content.strip()}

results_v2 = evaluate(
    extract_city_v2,
    data=DATASET_NAME,
    evaluators=[city_exact_match],
    experiment_prefix="city-extractor:v2-strict-prompt",
)

# 2) pairwise: 두 출력 중 더 나은 쪽 선정
def pref_shorter_city(runs: list, example) -> dict:
    # runs[i].outputs 는 None 일 수 있으므로 방어적으로 접근
    a = (runs[0].outputs or {}).get("city", "")
    b = (runs[1].outputs or {}).get("city", "")
    winner = 0 if len(a) <= len(b) else 1
    return {"key": "shorter_city", "scores": {runs[0].id: 1-winner, runs[1].id: winner}}

# evaluate_comparative 는 정확한 session name (또는 UUID) 가 필요하다.
# experiment_prefix 로 만든 session 의 자동 hash suffix 까지 포함된 전체 name 을 조회해서 전달.
# 또한 `experiments` 는 positional-only 인자이므로 kwarg 가 아닌 튜플로 넘긴다.
sessions = list(client.list_projects(reference_dataset_id=dataset.id))
v1_name = next(s.name for s in sessions if s.name.startswith("city-extractor:gpt-4.1-mini"))
v2_name = next(s.name for s in sessions if s.name.startswith("city-extractor:v2-strict-prompt"))
print(f"비교 대상: {v1_name}  vs  {v2_name}")

evaluate_comparative(
    (v1_name, v2_name),                # positional-only
    evaluators=[pref_shorter_city],
)

# 3) summary evaluator: 데이터셋 전체 exact-match 비율
def summary_accuracy(runs: list, examples: list) -> dict:
    total = len(runs)
    if total == 0:
        return {"key": "accuracy", "score": 0.0}
    correct = sum(
        1 for r, e in zip(runs, examples)
        if r.outputs and e.outputs and r.outputs.get("city") == e.outputs.get("city")
    )
    return {"key": "accuracy", "score": correct / total}

evaluate(
    extract_city,
    data=DATASET_NAME,
    evaluators=[city_exact_match],
    summary_evaluators=[summary_accuracy],
    experiment_prefix="city-extractor:with-summary",
)
print("pairwise + summary experiment 실행 완료")

## 8.03.7 Online evaluator — 프로덕션 trace 자동 평가

Offline experiment 는 배포 전 회귀 테스트용이고, 운영 중에는 **online evaluator** 로 실시간 feedback 을 붙인다. UI 흐름:

1. 프로젝트 -> **Evaluators** 탭 -> `+ Evaluator`
2. **LLM-as-judge** 선택, 평가 프롬프트 작성 (예: "응답이 사용자의 의도에 답하는가?")
3. 필터 지정 — 예: `has(tags, "env:prod")` 인 run 만
4. **Sampling rate** 를 0.1 로 두면 매칭 trace 의 10% 만 평가 — 비용 제어
5. 저장하면 신규 trace 에 자동으로 feedback key 가 붙기 시작

과거 trace 에도 소급 적용하려면 **Apply to past runs** 를 켜고 기간을 지정. 결과 feedback 은 대시보드·알림의 트리거가 되고, `05_production_monitoring.ipynb` 에서 알림 룰과 함께 다시 다룬다.

<!-- phase-c:embed -->
![Experiment 결과 + evaluator 차트](../assets/images/langsmith/03_datasets_and_evaluation/02_dataset_detail_examples.png)

*`weather-bot-qa` 의 Experiments 탭 — 상단 차트 3개 (Feedback 점수 `city_exact_match`/`city_non_empty`/`semantic_city_match`, Latency P50/P99, Tokens Input/Output) + 하단 실험 테이블 (#1 `city-extractor:gpt-4.1-mini-...` / #2 `city-extractor:v2-strict-prompt-...`). 각 실험은 3 runs 실행 후 점수 집계.*

## 체크리스트

- [ ] `client.create_dataset` + `client.create_examples` 로 수동 골든셋 생성
- [ ] `client.add_runs_to_dataset` 로 프로덕션 run 을 데이터셋에 이관
- [ ] Code evaluator 시그니처 `(inputs, outputs, reference_outputs) -> {"key","score"}` 확인
- [ ] LLM-as-judge 를 `with_structured_output` 으로 구조화
- [ ] `evaluate(...)` 로 experiment 실행 · UI 에서 이름으로 비교
- [ ] Pairwise (`evaluate_comparative`) 와 summary evaluator 의 시그니처 차이 이해
- [ ] Online evaluator 를 UI 에서 붙여 프로덕션 trace 에 자동 feedback

## 다음

- `04_prompt_hub.ipynb` — 이 실험에 쓴 프롬프트를 버전 관리하고 `prod` 태그로 배포하는 법

## 참고

- Evaluation 개요: https://docs.langchain.com/langsmith/evaluation
- Evaluate LLM application: https://docs.langchain.com/langsmith/evaluate-llm-application
- Pairwise evaluation: https://docs.langchain.com/langsmith/evaluate-pairwise
- Online evaluators: https://docs.langchain.com/langsmith/online-evaluations